# H&M GraphSAGE seed50 — 저장된 결과를 살리는 복구 전용 Colab

- 완료된 **M1·M2 결과는 그대로 재사용**하고 재학습·재평가하지 않습니다.
- 미완료 M3는 검증된 전체 체크포인트의 다음 epoch부터 이어서 학습한 뒤 M4를 진행합니다.
- 이전 9epoch는 복구 파일이 없어 재시작 승인 후 1epoch를 계산했습니다. 현재 1epoch 파일은 기존 런타임에 있습니다. **새 런타임으로 옮기기 전에 다운로드 또는 Drive 서버 저장 확인이 필수**입니다.
- 기존 코드 커밋·출력 경로·seed50·300epoch·학습/평가 조건을 유지합니다. M5나 다른 시드를 실행하지 않습니다.
- Google Drive 업로드 제한(403)이 남아 있으면 안전하게 중단합니다. 새 노트북 생성만으로 계정 제한이 해결되지는 않습니다.
- 기존 실행과 동시에 실행하지 마세요. 기존 런타임 종료 전 체크포인트를 확보하세요.
- 개인 Drive / L4 고용량 RAM. 매epoch 전체 가중치·Adam·난수 저장, 서버 해시 확인, 기존 최신·직전 보존 정책 유지.

## 1. 기존 환경 및 데이터 준비
아래 두 셀은 기존 노트북과 동일합니다. 고비용 학습을 시작하지 않습니다.


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import subprocess, sys, json, hashlib, time, uuid, shutil
import torch
assert torch.cuda.is_available(), 'L4 GPU 런타임으로 설정하세요.'
assert 'L4' in torch.cuda.get_device_name(0), '기존 실험과 동일한 L4로 실행하세요.'
drive.mount('/content/drive')
auth.authenticate_user()  # 두 승인 모두 개인계정 선택
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default()
drive_api = build('drive','v3',credentials=credentials,cache_discovery=False)
owner = drive_api.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
assert owner == 'azsabae@gmail.com', f'개인 Drive가 아닙니다: {owner}'
SOURCE_COMMIT = '17aff7a9b3a6b12448fa4fbca5fedd4e6977b30b'
REPO = Path('/content/clv-hm-graphsage-core4-s50-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == SOURCE_COMMIT
sys.path.insert(0,str(REPO))
import clv_hm_graphsage_core4_seed50 as screen
import clv_personal_graphsage_checkpoint as storage
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '이 런타임에 이전 코드가 남아 있습니다.'
print('개인 Drive 여유 GB:', storage.backend().check_space(drive_api,initial=True)/1e9)
DATA = Path('/content/drive/MyDrive/논문/data')
OUT = DATA / 'results_v3_hm_graphsage_core4_seed50_personal_v1'
OUT.mkdir(parents=True,exist_ok=True)
# 기존 입력 원본의 소유자와 바이트 해시로 두 연결이 같은 계정인지 확인한다.
p = DATA / 'raw/hm/articles.csv'
assert p.is_file(), f'입력 원본 없음: {p}'
h = hashlib.md5()
with p.open('rb') as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
        h.update(block)
meta = drive_api.files().get(fileId='1wf9deu3byBxY6Kk3M3v4syK6TWK8hWeY', fields='id,name,md5Checksum,owners(emailAddress,me),trashed').execute()
assert not meta.get('trashed') and meta.get('md5Checksum') == h.hexdigest() and any(o.get('me') and o.get('emailAddress') == owner for o in meta.get('owners', [])), '마운트와 API의 개인 Drive 일치 확인 실패'
print('개인 Drive 원본과 계정 일치 확인 완료')
print('GPU:', torch.cuda.get_device_name(0), '| seed50 M1~M4 | 출력:', OUT)


## 1. 입력 준비 — 학습 없음
개인 Drive의 원본 두 파일을 바이트 그대로 Colab 로컬 디스크에 복사합니다.
기존 참조 결과 폴더가 있으면 그대로 읽고, 없으면 아래 이름의 ZIP을 references 폴더에서 사용합니다.
ZIP도 없으면 **작은 결과 ZIP 두 개만** 업로드하도록 요청합니다. 원본 데이터 재변환·재학습은 하지 않습니다.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024**2),b''): h.update(block)
    return h.hexdigest()
EXPECTED = {
 'transactions_train.parquet':'0d90884fc1bc400161e6aeba238cba94183998bedfeb4ebbf4962e17d0c8489b',
 'articles.csv':'25f21882053517f7684ea43f524225a9481a7bb9aab9cf0abfc7fe8c4576a72b'}
LOCAL = Path('/content/hm_graphsage_seed50_inputs')
LOCAL.mkdir(exist_ok=True)
for key,name in [('tx_path','transactions_train.parquet'),('item_meta_path','articles.csv')]:
    source=DATA/'raw/hm'/name
    target=LOCAL/name
    if not target.is_file() or sha256(target)!=EXPECTED[name]:
        assert source.is_file(), f'기존 개인 Drive 원본 파일 없음: {source}'
        temporary=target.with_suffix(target.suffix+'.partial')
        shutil.copyfile(source,temporary)
        assert sha256(temporary)==EXPECTED[name], f'원본 바이트 불일치: {name}'
        temporary.replace(target)
    screen.base.v3.SCHEMA['hm'][key]=str(target)
defaults=screen.configure()
ref_specs=[
 (Path(defaults.m2_reference),'clv_m2_adaptive_hm_seed49_lastweek_results.zip'),
 (Path(defaults.m4_reference),'clv_hm_m4c_lastweek_seed49_results.zip')]
refs=[]
for directory,name in ref_specs:
    archive=DATA/'references'/name
    refs.append(directory if directory.is_dir() else archive)
missing=[p for p in refs if not p.exists()]
if missing:
    from google.colab import files
    print('다음 기존 결과 ZIP만 업로드하세요:', [p.name for p in missing])
    uploaded=files.upload()
    for p in missing:
        assert p.name in uploaded, f'누락: {p.name}'
        p.parent.mkdir(parents=True,exist_ok=True)
        p.write_bytes(uploaded[p.name])
cfg=screen.configure(out_dir=str(OUT),m2_reference=str(refs[0]),m4_reference=str(refs[1]))
assert cfg.seeds==(50,) and cfg.epochs==300
# Reuse a verified preparation in this runtime: rebuilding it duplicates large GPU graphs.
if 'prepared' in globals():
    from dataclasses import asdict
    assert asdict(prepared['cfg']) == asdict(cfg), '기존 준비 설정이 다릅니다. 저장 상태 확인 후 새 런타임을 사용하세요.'
    assert prepared['revision'] == SOURCE_COMMIT and prepared['input_hash'] == '50f1fc88d1144b2f2e739d4956536560ccc6fe444ba3fd10c55276d8cbdf762c', '기존 준비의 코드 또는 입력이 다릅니다.'
    assert all(k in prepared for k in ('graphsage_binary_adj','graphsage_m3_adj','references','config_hash','run_dir'))
    assert getattr(screen.SafeProgressStore, '_personal_storage_installed', False), '기존 저장 설정을 확인하세요.'
    checkpoint_record = json.loads((prepared['run_dir']/'personal_checkpoint_storage.json').read_text())
    assert checkpoint_record['account'] == 'azsabae@gmail.com'
    checkpoint_folder = checkpoint_record['folder_id']
    print('기존 준비 데이터·저장 설정 재사용: GPU 그래프를 다시 만들지 않습니다.', flush=True)
else:
    prepared=screen.prepare(cfg)
    checkpoint_folder=storage.install(screen,drive_api,prepared)
assert set(prepared['data']['splits'])=={'test'}
assert prepared['data'].get('loss_w') is None
print('사전점검 완료: M1~M4 / seed50 / 300epoch')
print('결과 폴더:', prepared['run_dir'])
print('체크포인트 폴더:', 'https://drive.google.com/drive/folders/'+checkpoint_folder)


## 2. M1·M2 완료 검증 및 M3 복구본 확보
서버 저장이 확인되기 전에는 다음 학습셀로 진행하지 않습니다.


In [ ]:
from dataclasses import asdict
import os

assert prepared['config_hash'] == 'bd7e0acfb22f', '기존 실행과 설정이 다릅니다'
assert cfg.seeds == (50,) and cfg.epochs == 300
assert screen.ORDER == ('M1','M2','M3','M4')
assert all(screen.completed(prepared,m) is not None for m in ('M1','M2')), 'M1/M2 완료 원본부터 복원하세요'
run_root = Path(prepared['run_dir'])
local_root = Path('/content/graphsage_seed50_core_checkpoints') / prepared['config_hash']
local_root.mkdir(parents=True, exist_ok=True)
backend = storage.backend()
personal_store = screen.SafeProgressStore
original_store = screen._personal_original_store

def checked_epoch(path, ident):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    assert payload.get('identity') == asdict(ident), '다른 실행/모형/시드의 파일입니다'
    epoch = payload.get('epoch')
    assert type(epoch) is int and 1 <= epoch <= 300
    assert [x.get('epoch') for x in payload.get('history',[])] == list(range(1,epoch+1))
    assert not payload.get('completed_weights_only'), '재개용 전체 상태가 아닙니다'
    for key in ('model_state','optimizer_state','numpy_rng_state','torch_rng_state','cuda_rng_state'):
        assert key in payload, '재개 상태 누락: '+key
    return epoch

def secure_local(store):
    path = store.latest_checkpoint
    epoch = checked_epoch(path, store.identity)
    # 동일 epoch의 다른 해시를 덮어쓰지 않으며, 서버 크기/MD5/SHA 메타 검증 후 반환합니다.
    try:
        receipt = backend.publish(drive_api, checkpoint_folder, path, store.identity, epoch)
    except Exception:
        print('Drive 저장 실패: 학습을 시작하지 않습니다. 로컬 복구본 보존:', path, flush=True)
        print('새 노트북도 계정 업로드 제한을 해제하지는 못합니다. 런타임 종료 전 이 파일을 다운로드하세요.',flush=True)
        raise
    print(f'원격 복구본 확인: {store.identity.model_id} epoch{epoch} / {receipt["size"]} bytes',flush=True)
    return epoch

class RecoveryStore(personal_store):
    def restore_epoch(self, model, optimizer, rng):
        if self.latest_checkpoint.is_file():
            local_epoch = checked_epoch(self.latest_checkpoint, self.identity)
            remote = backend.rows_for(drive_api, checkpoint_folder, self.identity)
            if not remote or local_epoch >= int(remote[0]['appProperties']['clv_epoch']):
                secure_local(self)
                return original_store.restore_epoch(self, model, optimizer, rng)
        if self.identity.model_id == screen.identity(prepared,'M3').model_id:
            assert backend.rows_for(drive_api,checkpoint_folder,self.identity), 'M3 복구본 없음: 자동 처음부터 학습 금지'
        return super().restore_epoch(model, optimizer, rng)

# 미완료 M3 복구본만 사용. 새 런타임에서는 Mac에 보존한 본인 체크포인트를 업로드할 수 있습니다.
if screen.completed(prepared,'M3') is None:
    m3_store = RecoveryStore(run_root/'progress', screen.identity(prepared,'M3'))
    local = m3_store.latest_checkpoint
    remote = backend.rows_for(drive_api,checkpoint_folder,m3_store.identity)
    mounted = run_root/'checkpoints_mount_recovery'/local.name
    if not local.is_file() and mounted.is_file():
        checked_epoch(mounted,m3_store.identity)
        temp = local.with_suffix('.recovery-copy')
        shutil.copyfile(mounted,temp)
        assert sha256(temp) == sha256(mounted)
        os.replace(temp,local)
    if not local.is_file() and not remote:
        from google.colab import files
        print('기존 런타임에서 받은 M3 seed50 전체 체크포인트 .pt를 선택하세요. 다른 시드 파일 사용 금지.')
        uploaded = files.upload()
        assert len(uploaded) == 1
        source = Path(next(iter(uploaded)))
        checked_epoch(source,m3_store.identity)
        shutil.copyfile(source,local)
        assert sha256(source) == sha256(local)
        del uploaded
    if local.is_file():
        local_epoch=checked_epoch(local,m3_store.identity)
        if not remote or local_epoch >= int(remote[0]['appProperties']['clv_epoch']):
            secure_local(m3_store)
    else:
        print('M3 원격 복구본 존재: 학습셀에서 해시·전체 상태 검증 후 재개합니다.')
screen.SafeProgressStore = RecoveryStore
print('준비 완료: M1/M2 재사용, M3 이어서 → M4. 매epoch 원격 저장 확인 유지.')


## 2. M1 → M2 → M3 → M4 학습
첫3epoch의 첫5배치 계측과 각 epoch의 학습·저장 시간을 기록합니다.
고정 입력 준비는 한 번만 수행합니다. 실제 시간 단축률은 GPU 실행 후 확인합니다.
오류 시 결과 패킷은 미완료(partial)로 보관하며 완료로 표시하지 않습니다.

In [ ]:
assert getattr(screen.SafeProgressStore,'_personal_storage_installed',False)
assert screen.ORDER==('M1','M2','M3','M4') and cfg.seeds==(50,)
try:
    result=screen.run(prepared)
    print(json.dumps(result['reading'],ensure_ascii=False,indent=2))
finally:
    output=screen.export_zip(prepared,'/content/hm_graphsage_core4_seed50_results.zip')
    backup=prepared['run_dir']/Path(output).name
    shutil.copyfile(output,backup)
    assert sha256(output)==sha256(backup)
    print('현재까지 결과 ZIP:', backup)


## 3. 저장 결과와 시간 확인 — 추가 학습·평가 없음
M1 대비 동일 seed50 비교입니다. 전체142지표·진단·학습이력을 ZIP에 보존합니다.
학습 중에는 이 셀을 실행하지 마세요.

In [ ]:
import pandas as pd
from IPython.display import display
result=screen.report(prepared)
display(result['absolute'].set_index('model')[list(screen.CORE)].T.reindex(columns=screen.ORDER))
comparison=result['comparison']
if not comparison.empty:
    display(comparison.query("reference == 'M1'").pivot(index='metric',columns='model',
        values='relative_change_pct').reindex(index=screen.CORE))
paths=sorted((prepared['run_dir']/'progress/timing').glob('*.csv'))
if paths:
    timing=pd.concat([pd.read_csv(p) for p in paths],ignore_index=True)
    display(timing.groupby('model_id')[['train_seconds','save_seconds']].mean())
print(json.dumps(result['reading'],ensure_ascii=False,indent=2))
print('seed50 1시드 결과. M5_A~D는 이 노트북에서 실행하지 않습니다.')


## 4. 결과 다운로드
ZIP은 성능·설정·진단·학습이력만 포함합니다. 대용량 재개 체크포인트는 별도 개인 Drive 폴더에 유지합니다.
다운로드가 멈추면 출력된 Drive 결과 폴더에서 ZIP을 직접 받으세요.

In [ ]:
from google.colab import files
output=screen.export_zip(prepared,'/content/hm_graphsage_core4_seed50_results.zip')
backup=prepared['run_dir']/Path(output).name
shutil.copyfile(output,backup)
assert sha256(output)==sha256(backup)
print('Drive ZIP:',backup)
files.download(output)
